# 00 - Setup

This notebook prepares the local workshop environment:
- Create and configure `.env` with your API key
- Start Qdrant in Docker and verify it is reachable
- Test HPI API connection (Embeddings + Generation)
- Check data files (Workshop 3: CSV + PDF)

> Run this notebook once **before** starting the workshop notebooks (w2_01–w2_04, w3_01–w3_04).

In [1]:
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'docker-compose.yml').exists():
            return candidate
    raise RuntimeError('Could not find repository root (docker-compose.yml missing).')

REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOKS_DIR = REPO_ROOT / 'notebooks'
ENV_PATH = NOTEBOOKS_DIR / '.env'
print('Repo root:', REPO_ROOT)
print('Notebooks dir:', NOTEBOOKS_DIR)
print('.env path:', ENV_PATH)


Repo root: /Users/felixboelter/Documents/GitHub/workshop-ragV2
Notebooks dir: /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks
.env path: /Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/.env


## 1) Configure `.env` (API key + API base URL)

Set your API credentials in `notebooks/.env`.

Expected keys for notebook experiments:
- `OPENAI_API_KEY`
- `OPENAI_API_BASE` (here: `https://api.aisc.hpi.de/`)

> Do not commit real secrets to git.


In [ ]:
from pathlib import Path

template = '''# Notebook environment
OPENAI_API_KEY=your_openai_api_key_here

# API base URL (both names for compatibility)
OPENAI_API_BASE=https://api.aisc.hpi.de/

# Embedding defaults (can be overridden)
OPENAI_EMBEDDING_MODEL=octen-embedding-8b
OPENAI_EMBEDDING_DIM=4096

# Optional Qdrant override (defaults used in this notebook)
QDRANT_HOST=localhost
QDRANT_PORT=6333
'''.strip() + '\n'

if not ENV_PATH.exists():
    ENV_PATH.write_text(template, encoding='utf-8')
    print(f'Created {ENV_PATH}')
else:
    print(f'{ENV_PATH} already exists')

print(
    'Ensure OPENAI_API_KEY is set. '
    'OPENAI_BASE_URL/OPENAI_API_BASE and embedding vars have defaults.'
)


/Users/felixboelter/Documents/GitHub/workshop-ragV2/notebooks/.env already exists
Ensure OPENAI_API_KEY and OPENAI_API_BASE are set before running later notebooks.


In [3]:
import os

# lightweight .env reader (no extra dependency required)
for line in ENV_PATH.read_text(encoding='utf-8').splitlines():
    line = line.strip()
    if not line or line.startswith('#') or '=' not in line:
        continue
    key, value = line.split('=', 1)
    os.environ[key.strip()] = value.strip()

api_key = os.getenv('OPENAI_API_KEY', '')
api_base = os.getenv('OPENAI_API_BASE', '')
if not api_key or api_key == 'your_openai_api_key_here':
    raise ValueError('OPENAI_API_KEY is not configured in notebooks/.env yet.')
if not api_base:
    raise ValueError('OPENAI_API_BASE is not configured in notebooks/.env yet.')

print('OPENAI_API_KEY is configured (value hidden).')
print('OPENAI_API_BASE:', api_base)


OPENAI_API_KEY is configured (value hidden).
OPENAI_API_BASE: https://api.aisc.hpi.de/


## 2) Start Qdrant in Docker

This starts (or reuses) a local `qdrant` container with persistent storage in `qdrant_storage/`.


In [4]:
import subprocess

storage_dir = REPO_ROOT / 'qdrant_storage'
storage_dir.mkdir(exist_ok=True)

docker_check = subprocess.run(['docker', '--version'], capture_output=True, text=True)
if docker_check.returncode != 0:
    raise RuntimeError('Docker is required to start Qdrant from this notebook.')

exists = subprocess.run(['docker', 'ps', '-aq', '-f', 'name=^qdrant$'], capture_output=True, text=True)
container_id = exists.stdout.strip()

if container_id:
    running = subprocess.run(['docker', 'ps', '-q', '-f', 'name=^qdrant$'], capture_output=True, text=True)
    if running.stdout.strip():
        print('Qdrant container is already running.')
    else:
        subprocess.run(['docker', 'start', 'qdrant'], check=True)
        print('Started existing qdrant container.')
else:
    subprocess.run([
        'docker', 'run', '-d',
        '--name', 'qdrant',
        '--restart', 'unless-stopped',
        '-p', '6333:6333',
        '-v', f'{storage_dir}:/qdrant/storage',
        'qdrant/qdrant:latest',
    ], check=True)
    print('Started new qdrant container.')

print('Qdrant dashboard: http://localhost:6333/dashboard')


Qdrant container is already running.
Qdrant dashboard: http://localhost:6333/dashboard


In [5]:
import json
from urllib.request import urlopen

with urlopen('http://localhost:6333') as resp:
    payload = json.loads(resp.read().decode('utf-8'))

print('Qdrant is reachable:')
print(payload)


Qdrant is reachable:
{'title': 'qdrant - vector search engine', 'version': '1.16.3', 'commit': 'bd49f45a8a2d4e4774cac50fa29507c4e8375af2'}


## Setup complete

All systems verified:
- `.env` with API credentials configured
- Qdrant running locally
- HPI API reachable (Embeddings + Generation)
- Data files present (Workshop 3)

Continue with the workshop notebooks:
- **Workshop 2 (RAG Fundamentals):** `w2_01_chunking_and_retrieval.ipynb`
- **Workshop 3 (RAG Evaluation):** `w3_01_intro_end_to_end.ipynb`

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    base_url=os.getenv('OPENAI_API_BASE'),
)

# Test embedding endpoint
embedding_model = os.getenv('OPENAI_EMBEDDING_MODEL', 'octen-embedding-8b')
emb_response = client.embeddings.create(
    input=['Test embedding connection'],
    model=embedding_model,
    encoding_format='float',
)
emb_dim = len(emb_response.data[0].embedding)
print(f'Embedding OK — model: {embedding_model}, dimensions: {emb_dim}')

In [ ]:
# Test generation endpoint
chat_model = 'gpt-oss-120b'

chat_response = client.chat.completions.create(
    model=chat_model,
    messages=[{'role': 'user', 'content': 'Antworte mit genau einem Wort: Funktioniert die Verbindung?'}],
    max_tokens=1000,
)
print(f'Generation OK — model: {chat_model}')
print(f'Response: {chat_response.choices[0].message.content}')

## 4) Check data files (Workshop 3)

Verifies that the ground-truth CSV and source PDF for Workshop 3 (RAG Evaluation) are present.

In [ ]:
import pandas as pd

DATA_DIR = NOTEBOOKS_DIR / 'data'
csv_path = DATA_DIR / 'GSKI_Fragen-Antworten-Fundstellen_123_Einfach.csv'
pdf_path = DATA_DIR / 'IT_Grundschutz_Kompendium_Edition2023.pdf'

assert csv_path.exists(), f'CSV not found: {csv_path}'
assert pdf_path.exists(), f'PDF not found: {pdf_path}'

df = pd.read_csv(csv_path, sep=';')
print(f'Ground truth CSV: {len(df)} rows, columns: {list(df.columns)}')
print(f'Source PDF: {pdf_path.name} ({pdf_path.stat().st_size / 1024 / 1024:.1f} MB)')
df.head(3)

## Next
Run the next notebook after this setup is complete:
- `01_chunking_and_retrieval.ipynb`
